In [2]:
# =============================================
# NOTEBOOK SETUP — Run this cell first
# =============================================
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import duckdb
from config.settings import DB_PATH

con = duckdb.connect(str(DB_PATH), read_only=True)

print("✅ Connected read-only!")
print("✅ DuckDB view is live!")

# Fixed count line
total_df = con.sql("SELECT COUNT(*) as total FROM silver_lineups").df()
total = total_df.iloc[0]['total']
print(f"✅ Total games in silver_lineups: {total:,}")

✅ Connected read-only!
✅ DuckDB view is live!
✅ Total games in silver_lineups: 64


In [7]:
# Most recent lineups with the new bbref columns
con.sql("""
    SELECT 
        game_date,
        away_team,
        home_team,
        away_starter_name,
        away_starter_primary_bbref_id,
        away_lineup_bbref_ids[0:3] AS first_three_away_bbref_ids,
        is_confirmed
    FROM silver_lineups 
    ORDER BY fetch_timestamp DESC 
    LIMIT 5
""").df()

,game_date,away_team,home_team,away_starter_name,away_starter_primary_bbref_id,first_three_away_bbref_ids,is_confirmed
0,2026-02-25,PIT,BOS,C. Mlodzinski,mlodzca01,"[manguja01, gonzada01, horwisp01]",True
1,2026-02-25,BAL,MIN,Albert Suarez,suareal01,"[taveral01, basalsa01, jacks07]",True
2,2026-02-25,DET,ATL,E. De Jesus,jesso01,"[meadole01, mcgunbi01, ]",True
3,2026-02-25,MIN,TB,C. MacLeod,mcleoji01,"[kreidry01, jenki04, lewisro02]",True
4,2026-02-25,NYY,TOR,Will Warren,warrewi01,"[grishtr01, judgeaa01, bellico01]",True


In [4]:
# Quick check that matching worked
con.sql("""
    SELECT 
        game_date,
        away_team,
        away_starter_name,
        away_starter_primary_bbref_id
    FROM silver_lineups 
    WHERE away_starter_primary_bbref_id IS NOT NULL
    LIMIT 8
""").df()


,game_date,away_team,away_starter_name,away_starter_primary_bbref_id
0,2026-02-25,PIT,C. Mlodzinski,mlodzca01
1,2026-02-25,BAL,Albert Suarez,suareal01
2,2026-02-25,DET,E. De Jesus,jesso01
3,2026-02-25,MIN,C. MacLeod,mcleoji01
4,2026-02-25,NYY,Will Warren,warrewi01
5,2026-02-25,HOU,Jason Alexander,alexaja01
6,2026-02-25,PHI,Seth Johnson,johnsse01
7,2026-02-25,MIL,Angel Zerpa,zerpaan01


In [8]:
# Quick validation — how many players matched?
con.sql("""
    SELECT 
        COUNT(CASE WHEN away_starter_primary_bbref_id IS NOT NULL THEN 1 END) as starters_matched,
        COUNT(*) as total_starters
    FROM silver_lineups
""").df()

,starters_matched,total_starters
0,64,64


In [16]:
# Most recent lineups with the new bbref columns
x = con.sql("""
    SELECT *
    FROM silver_lineups 
    ORDER BY fetch_timestamp DESC 
    LIMIT 5
""").df()
x

,fetch_timestamp,game_date,game_time,away_team,home_team,away_starter_name,away_starter_hand,home_starter_name,home_starter_hand,away_lineup,...,umpire,odds_line,odds_ou,away_starter_primary_bbref_id,home_starter_primary_bbref_id,away_lineup_bbref_ids,home_lineup_bbref_ids,day,month,year
0,2026-02-25 01:57:52.734099,2026-02-25,1:05 PM ET,PIT,BOS,C. Mlodzinski,R,Ranger Suarez,L,"[Jake Mangum, N. Gonzales, S. Horwitz, Nick Yo...",...,<NA>,"{'betmgm': None, 'composite': None, 'draftking...","{'betmgm': 8.5, 'composite': 8.5, 'draftkings'...",mlodzca01,suarera01,"[manguja01, gonzada01, horwisp01, yorkeni01, v...","[anthoer01, storytr01, duranja01, contrma01, a...",25,02,2026
1,2026-02-25 01:57:52.734099,2026-02-25,1:05 PM ET,BAL,MIN,Albert Suarez,R,Andrew Morris,R,"[L. Taveras, S. Basallo, J. Jackson, Coby Mayo...",...,<NA>,"{'betmgm': None, 'composite': None, 'draftking...","{'betmgm': None, 'composite': None, 'draftking...",suareal01,moorean02,"[taveral01, basalsa01, jacks07, mayoco01, kjer...","[buxtoby01, clemeko01, belljo02, larnatr01, ca...",25,02,2026
2,2026-02-25 01:57:52.734099,2026-02-25,1:05 PM ET,DET,ATL,E. De Jesus,L,Reynaldo Lopez,R,"[P. Meadows, K. McGonigle, W. Perez, Jahmai Jo...",...,<NA>,"{'betmgm': None, 'composite': None, 'draftking...","{'betmgm': 8.5, 'composite': 8.5, 'draftkings'...",jesso01,lopezre01,"[meadole01, mcgunbi01, , jonesja08, mckinza01,...","[acunaro01, profaju01, olsonma02, rileyau01, a...",25,02,2026
3,2026-02-25 01:57:52.734099,2026-02-25,1:05 PM ET,MIN,TB,C. MacLeod,L,Joe Boyle,R,"[R. Kreidler, W. Jenkins, Royce Lewis, Alan Ro...",...,<NA>,"{'betmgm': None, 'composite': None, 'draftking...","{'betmgm': None, 'composite': None, 'draftking...",mcleoji01,boylejo01,"[kreidry01, jenki04, lewisro02, rodenal01, wag...","[willibe03, diazya01, randajo01, caminju01, mu...",25,02,2026
4,2026-02-25 01:57:52.734099,2026-02-25,1:07 PM ET,NYY,TOR,Will Warren,R,Grant Rogers,R,"[T. Grisham, Aaron Judge, C. Bellinger, J. Chi...",...,<NA>,"{'betmgm': None, 'composite': None, 'draftking...","{'betmgm': 9.5, 'composite': 9.5, 'draftkings'...",warrewi01,robergr01,"[grishtr01, judgeaa01, bellico01, chishja01, g...","[strawmy01, , sanchto01, schneda01, jimenle01,...",25,02,2026
